# Correa overland · alertas predictivas

Dos reductores y seis sensores. Datos sintéticos horarios, abril–septiembre 2026.
Warning ≥16; trip ≥20; riesgo de trip en próximas 24 h. Unidad arbitraria.

Esta exploración abre el resultado ya ejecutado, sin necesidad de reentrenar. La última sección permite regenerar el proyecto. **El modelo no supera globalmente al warning en esta ejecución.**

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
root = Path('.')
df = pd.read_csv(root/'historico_6_meses.csv', parse_dates=['timestamp'])
metricas = json.loads((root/'metricas.json').read_text())
display(df.head())
print(f'{len(df):,} filas; 6 sensores; {df.trip.sum()} trips')

## Histórico y límites de protección
El máximo de cualquiera de los seis sensores determina warning o detención.

In [ ]:
display(Image(filename=str(root/'historico.png')))
display(df.groupby('estado').size().rename('muestras'))

## Evaluación temporal
Abril–julio: entrenamiento; agosto: validación; septiembre: prueba. Purga de 24 h entre conjuntos. Ventanas causales reiniciadas tras mantenimiento. Últimas 24 h sin etiqueta, porque falta futuro.

In [ ]:
display(pd.DataFrame({
    'Random Forest': metricas['ml_prueba'],
    'Warning 16': metricas['warning_16_prueba'],
    'Extrapolación lineal': metricas['lineal_prueba']
}).T)
display(pd.read_csv(root/'anticipacion_prueba.csv'))
display(pd.Series(metricas['metricas_horarias_prueba']))

## Un patrón que permite alertar antes de 16
En el evento 29, R2_S2 aumenta de manera sostenida y crecen otros sensores de R2. La alerta ML comienza 22 h antes del trip; el warning 15 h antes. Es un ejemplo favorable dentro de una prueba donde otros eventos son detectados más tarde.

In [ ]:
display(Image(filename=str(root/'evento_29.png')))

## Límite: falla sin precursor
La falla repentina del evento 30 no fue anticipada ni por ML ni por el warning.

In [ ]:
display(Image(filename=str(root/'evento_30.png')))
display(pd.read_csv(root/'importancia_variables.csv').head(10))

## Reproducir entrenamiento y simulación
Descomentar para ejecutar. El umbral se ajusta en agosto y luego permanece fijo. No se usa septiembre para seleccionar umbrales. Para probar otra semilla o frecuencia, cambiar parámetros y mantener una evaluación temporal.

In [ ]:
from overland import run
# datos_nuevos, resultados = run('resultados_notebook', seed=42, horizon=24, step_minutes=60)

## Aplicar a un nuevo histórico
CSV con timestamp UTC, seis sensores y operando (0/1). Ejecutar desde terminal:
```bash
python predecir.py nuevo_historico.csv --model modelo.joblib --out predicciones_nuevas.csv
```
Leer INFORME.md antes de extrapolar los resultados a datos reales.